In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from dataclasses import dataclass

@dataclass
class Context:
    user_name: str
    is_premium: bool = False

In [6]:
from langchain.agents.middleware import wrap_model_call
from langchain.chat_models import init_chat_model

@wrap_model_call
def dynamic_model_selector(request, handler):
    user_name =request.runtime.context.user_name
    is_premium = request.runtime.context.is_premium
    
    if is_premium:
        model_name = "google_genai:gemini-3.5-flash"
        print(f"{user_name} 사용자는 프리미엄 회원입니다. gemini-3.5-flash를 호출합니다.")
    else:
        model_name = "google_genai:gemini-2.5-flash"
        print(f"{user_name} 사용자는 일반 회원입니다. gemini-2.5-flash를 호출합니다.")

    new_model = init_chat_model(model_name)
    new_request = request.override(model=new_model)

    return handler(new_request)

In [7]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite", 
    tools=[],
    middleware=[dynamic_model_selector],
    context_schema=Context
)

In [8]:
agent.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요"}]},
    context = Context(user_name="김일남", is_premium=True)
)

김일남 사용자는 프리미엄 회원입니다. gemini-3.5-flash를 호출합니다.


{'messages': [HumanMessage(content='안녕하세요', additional_kwargs={}, response_metadata={}, id='984e1226-a42a-4464-b06e-44b430294b41'),
  AIMessage(content=[{'type': 'text', 'text': '안녕하세요! 반갑습니다. 오늘 어떤 도움이 필요하신가요? 무엇이든 편하게 물어보세요!', 'extras': {'signature': 'EvQGCvEGAQw51sdbxIzGRW85KRsk5zt1aGa1zOg7mzhP1CwyEaQPUQHdvfCjc/CnkRs3LnBFmOLCYrKlmbhPan7Ozqy7cWHLjYuKakRi5N3bxZwgYn9WQyhyQHjRwq2y9hCVgEU2OAexfoGBIBKhQkA0rRqrjkyKtX7gQOv9D0eIyIvalhWG+7Y122FZ5Ehj+9vWIHaHpjn1g8LUiJUv7hfGmYB/jH8vvYW5ZprkmCgk+mdC019JNJqnCczn1xNpxvSwPf5P/kpjO+g0z5kdcNeD8tcaIsUfnfEsdgXsW+62t1mYEKhxTVml/k2gm2NIRHJ7ynVar/g6a0Ww8liyDZKvW9AkjWw1W/n/RHftv2zUizPdGRJXiK69Gp7os68PV67St3AbhlewXYAoJ+dIlwDWGi4BHw+adthDNtYuSOI4K6jrkaADlttpAHJP0/FVTpN2yYTR52/zBH3ig83obsXAVFIXm5qV1TOM5agaXrJRxgNy2S0oT4ST83aGadNKKHl/eOPUY/cS4qKZKnBM8XSHKLfSYIKn8VKv+apW8ePiiaz5GGmygvHKA/VC9+i10QRobo54Q5pLqhra86kFPjx9O94QiuIXLiZL/zE9yGnKYoOOJMjN6vMJBtMJzlnkTyl+1ngbx+4uABgR+WlOAOe0lqEb6s/yiaalss6345yzepro8OZCdBGdyR6oZ+KBrIyDemDtp8gsmqmVl04C/kdqJRzF+pmjwDm0UgIl6n

In [9]:
agent.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요"}]},
    context = Context(user_name="김이남", is_premium=False)
)

김이남 사용자는 일반 회원입니다. gemini-2.5-flash를 호출합니다.


{'messages': [HumanMessage(content='안녕하세요', additional_kwargs={}, response_metadata={}, id='86dfbb19-512a-4a4b-a3f1-7d64c1bcdbf2'),
  AIMessage(content='안녕하세요! 반갑습니다. 😊', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7ab0-4ddb-7922-ab3d-d33b798bbf0d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 35, 'total_tokens': 37, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 30}})]}

---

In [10]:
from langchain.agents.middleware import wrap_model_call
from langchain.chat_models import init_chat_model

@wrap_model_call
def dynamic_model_selector(request, handler):
    last_msg = request.messages[-1].content if request.messages else ""
    msg_len = len(last_msg)

    print("msg_len:", msg_len)
    
    if msg_len < 10:
        model_name = "google_genai:gemini-2.5-flash-lite"
    elif msg_len < 30:
        model_name = "google_genai:gemini-2.5-flash"
    else:
        model_name = "google_genai:gemini-3.5-flash"

    new_model = init_chat_model(model_name)
    new_request = request.override(model=new_model)

    return handler(new_request)

In [11]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite", 
    tools=[],
    middleware=[dynamic_model_selector],
)

In [12]:
agent.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요"}]},
)

msg_len: 5


{'messages': [HumanMessage(content='안녕하세요', additional_kwargs={}, response_metadata={}, id='7e4151bf-bb83-4f31-a4ed-7b4c5cef6ddc'),
  AIMessage(content='안녕하세요! 무엇을 도와드릴까요? 😊', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e7ab2-babf-79b2-80d2-5ce218d49017-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 9, 'total_tokens': 11, 'input_token_details': {'cache_read': 0}})]}

In [13]:
agent.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요. 저는 김일남입니다. 랭체인에 대해서 궁금한게 많습니다."}]},
)

msg_len: 37


{'messages': [HumanMessage(content='안녕하세요. 저는 김일남입니다. 랭체인에 대해서 궁금한게 많습니다.', additional_kwargs={}, response_metadata={}, id='fbaa5033-ed11-4cee-b143-edecc715a316'),
  AIMessage(content=[{'type': 'text', 'text': '안녕하세요, 일남 님! 반갑습니다. \n\n요즘 인공지능 분야에서 가장 핫한 도구 중 하나인 **랭체인(LangChain)**에 관심을 가지게 되셨군요. 랭체인은 처음 접하면 개념이 조금 생소할 수 있지만, 이해하고 나면 LLM(대형 언어 모델)을 활용해 정말 다양하고 강력한 프로그램을 만들 수 있게 해주는 멋진 도구입니다.\n\n쉽게 설명해 드리자면, 랭체인은 **GPT 같은 언어 모델(LLM)에 \'날개\'를 달아주는 도구**라고 생각하시면 됩니다. \n\n일반 GPT는 과거 데이터만 알고 있거나 외부 도구를 쓰지 못하지만, 랭체인을 사용하면 다음과 같은 일들이 가능해집니다.\n1. **회사 내부 문서나 PDF 책을 읽고** 그 내용을 바탕으로 답변하는 챗봇 만들기 (RAG 기술)\n2. 이전 대화 내용을 기억하는 **메모리 기능** 추가하기\n3. AI가 스스로 판단해서 인터넷 검색을 하거나 계산기를 쓰는 등 **도구(Agent) 사용하게 만들기**\n\n일남 님께서는 랭체인에 대해 어떤 부분이 가장 궁금하신가요? 아래 예시 중에서 관심 있는 주제를 골라 말씀해 주셔도 좋고, 개인적으로 구상 중이신 아이디어를 말씀해 주셔도 좋습니다.\n\n1. **개념 이해:** "랭체인이 정확히 왜 필요한가요? 그냥 API 쓰는 거랑 뭐가 다른가요?"\n2. **핵심 기능:** "체인(Chain), 에이전트(Agent), 프롬프트 템플릿 같은 용어들이 뭔지 궁금해요."\n3. **실전 활용:** "제 PDF 파일이나 메모장 데이터를 학습시켜서 답변하는 챗봇을 만들고 싶어요."\n4. **시작하기:*